# 02 — All-Lecture RAG MVP: Retrieval → Reranking → RAG

# Step 0: Setup


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
!pip install -q     langchain     langchain-core     langchain-chroma     langchain-huggingface     langchain-openai     sentence-transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

In [ ]:
from pathlib import Path
import json
import os

PROJECT_ROOT = Path("/content/drive/MyDrive/AI_Engineering_Final_Project")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TEXT_DIR = PROCESSED_DIR / "transcripts"
METADATA_DIR = PROCESSED_DIR / "metadata"

RAG_DIR = PROJECT_ROOT / "rag"
CHROMA_DIR = RAG_DIR / "chroma_l01_l34"
RAG_CONFIG_PATH = (RAG_DIR / "all_lectures_rag_config.json")

RAG_DIR.mkdir(parents=True, exist_ok=True)
#CHROMA_DIR.mkdir(parents=True, exist_ok=True)

LECTURE_START = 1
LECTURE_END = 34

LECTURE_NUMBERS = list(range(LECTURE_START, LECTURE_END + 1))

print("Lectures:", LECTURE_NUMBERS)
print("Chroma directory:", CHROMA_DIR)
print("RAG config:", RAG_CONFIG_PATH)


# Step 1: Load Timestamped Transcripts + Lecture Metadata

The same code loads every lecture from `LECTURE_START` to `LECTURE_END`.

Expected files:

- `lecture_01_stt.json`, `lecture_02_stt.json`, ...
- `lecture_01_metadata.json`, `lecture_02_metadata.json`, ...


In [ ]:
lecture_records = []

for lecture_number in LECTURE_NUMBERS:
    lecture_id = f"lecture_{lecture_number:02d}"

    stt_path = TEXT_DIR / f"{lecture_id}_stt.json"
    metadata_path = METADATA_DIR / f"{lecture_id}_metadata.json"

    assert stt_path.exists(), f"Missing STT file: {stt_path}"
    assert metadata_path.exists(), f"Missing metadata file: {metadata_path}"

    with open(stt_path, "r", encoding="utf-8") as f:
        stt_segments = json.load(f)

    with open(metadata_path, "r", encoding="utf-8") as f:
        lecture_metadata = json.load(f)

    lecture_records.append({
        "lecture_number": lecture_number,
        "lecture_id": lecture_id,
        "stt_segments": stt_segments,
        "metadata": lecture_metadata,
    })

    print(
        f"Loaded Lecture {lecture_number}: "
        f"{lecture_metadata['lecture_title']} | "
        f"{len(stt_segments)} STT segments"
    )

print("\nTotal lectures loaded:", len(lecture_records))


# Validate that Notebook 1 supplied real MIT OCW lecture titles
# before these records are embedded into the full-course Chroma index.
generic_title_lectures = []

for record in lecture_records:
    number = record["lecture_number"]
    title = str(record["metadata"].get("lecture_title", "")).strip()

    if not title or title.lower() == f"lecture {number}".lower():
        generic_title_lectures.append(number)

assert not generic_title_lectures, (
    "Generic/missing lecture titles found for: "
    f"{generic_title_lectures}. "
    "Run the updated Notebook 1 first so it fetches official MIT OCW titles."
)

print("✅ Real lecture titles found for all loaded lectures.")


Loaded Lecture 1: The geometry of linear equations | 280 STT segments
Loaded Lecture 2: Elimination with matrices | 733 STT segments
Loaded Lecture 3: Multiplication and inverse matrices | 661 STT segments
Loaded Lecture 4: Factorization into A = LU | 233 STT segments
Loaded Lecture 5: Transposes, permutations, spaces R^n | 682 STT segments
Loaded Lecture 6: Column space and nullspace | 658 STT segments
Loaded Lecture 7: Solving Ax = 0: pivot variables, special solutions | 647 STT segments
Loaded Lecture 8: Solving Ax = b: row reduced form R | 680 STT segments
Loaded Lecture 9: Independence, basis, and dimension | 731 STT segments
Loaded Lecture 10: The four fundamental subspaces | 714 STT segments
Loaded Lecture 11: Matrix spaces; rank 1; small world graphs | 437 STT segments
Loaded Lecture 12: Graphs, networks, incidence matrices | 722 STT segments
Loaded Lecture 13: Quiz 1 review | 601 STT segments
Loaded Lecture 14: Orthogonal vectors and subspaces | 647 STT segments
Loaded Lecture

In [ ]:
def seconds_to_timestamp(seconds):
    seconds = int(seconds)
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60

    if hours > 0:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"

    return f"{minutes:02d}:{secs:02d}"


def build_timestamp_link(video_url, seconds):
    return f"{video_url}#t={int(seconds)}"


# Step 2: Create Timestamp-Aware Chunks


STT produces many small segments. Those are usually too small for good semantic retrieval. If I embed them individually, retrieval may return only a fragment.
So I combine neighboring segments.


In [ ]:
def create_timestamp_chunks(
    segments,
    max_chars=1200,
    overlap_segments=1,
):
    chunks = []
    current = []
    current_chars = 0

    for segment in segments:
        text = segment["text"].strip()

        if not text:
            continue

        if current and current_chars + len(text) > max_chars:
            chunks.append({
                "text": " ".join(s["text"].strip() for s in current),
                "start_seconds": current[0]["start_seconds"],
                "end_seconds": current[-1]["end_seconds"],
            })

            current = current[-overlap_segments:] if overlap_segments else []
            current_chars = sum(len(s["text"]) for s in current)

        current.append(segment)
        current_chars += len(text)

    if current:
        chunks.append({
            "text": " ".join(s["text"].strip() for s in current),
            "start_seconds": current[0]["start_seconds"],
            "end_seconds": current[-1]["end_seconds"],
        })

    return chunks


CHUNK_SIZE = 1200

for record in lecture_records:
    record["rag_chunks"] = create_timestamp_chunks(
        record["stt_segments"],
        max_chars=CHUNK_SIZE,
        overlap_segments=1,
    )

    print(
        f"Lecture {record['lecture_number']}: "
        f"{len(record['rag_chunks'])} chunks"
    )

print(
    "Total RAG chunks:",
    sum(len(record["rag_chunks"]) for record in lecture_records),
)


Lecture 1: 25 chunks
Lecture 2: 25 chunks
Lecture 3: 23 chunks
Lecture 4: 31 chunks
Lecture 5: 24 chunks
Lecture 6: 23 chunks
Lecture 7: 22 chunks
Lecture 8: 23 chunks
Lecture 9: 25 chunks
Lecture 10: 25 chunks
Lecture 11: 23 chunks
Lecture 12: 24 chunks
Lecture 13: 25 chunks
Lecture 14: 24 chunks
Lecture 15: 24 chunks
Lecture 16: 25 chunks
Lecture 17: 25 chunks
Lecture 18: 25 chunks
Lecture 19: 31 chunks
Lecture 20: 27 chunks
Lecture 21: 27 chunks
Lecture 22: 30 chunks
Lecture 23: 27 chunks
Lecture 24: 27 chunks
Lecture 25: 23 chunks
Lecture 26: 24 chunks
Lecture 27: 27 chunks
Lecture 28: 23 chunks
Lecture 29: 20 chunks
Lecture 30: 26 chunks
Lecture 31: 26 chunks
Lecture 32: 26 chunks
Lecture 33: 23 chunks
Lecture 34: 22 chunks
Total RAG chunks: 850


### Selected configuration

For this expansion, reuse the Lecture 1 decisions without rerunning the configuration benchmarks:

- `CHUNK_SIZE = 1200`
- `RETRIEVAL_K = 8`

The dedicated benchmark stage comes later in the project checklist.


# Step 3: Create LangChain Documents


This is where RAG data model becomes structured.


In [ ]:
from langchain_core.documents import Document

documents = []

for record in lecture_records:
    lecture_number = record["lecture_number"]
    lecture_metadata = record["metadata"]

    for i, chunk in enumerate(record["rag_chunks"]):
        documents.append(
            Document(
                page_content=chunk["text"],
                metadata={
                    "chunk_id": f"L{lecture_number:02d}_C{i:03d}",
                    "lecture_id": record["lecture_id"],
                    "lecture_number": lecture_number,
                    "lecture_title": lecture_metadata["lecture_title"],
                    "start_seconds": chunk["start_seconds"],
                    "end_seconds": chunk["end_seconds"],
                    "video_url": lecture_metadata["video_url"],
                },
            )
        )

print("Total Documents:", len(documents))

for lecture_number in LECTURE_NUMBERS:
    count = sum(
        1 for doc in documents
        if doc.metadata["lecture_number"] == lecture_number
    )
    print(f"Lecture {lecture_number}: {count} documents")


Total Documents: 850
Lecture 1: 25 documents
Lecture 2: 25 documents
Lecture 3: 23 documents
Lecture 4: 31 documents
Lecture 5: 24 documents
Lecture 6: 23 documents
Lecture 7: 22 documents
Lecture 8: 23 documents
Lecture 9: 25 documents
Lecture 10: 25 documents
Lecture 11: 23 documents
Lecture 12: 24 documents
Lecture 13: 25 documents
Lecture 14: 24 documents
Lecture 15: 24 documents
Lecture 16: 25 documents
Lecture 17: 25 documents
Lecture 18: 25 documents
Lecture 19: 31 documents
Lecture 20: 27 documents
Lecture 21: 27 documents
Lecture 22: 30 documents
Lecture 23: 27 documents
Lecture 24: 27 documents
Lecture 25: 23 documents
Lecture 26: 24 documents
Lecture 27: 27 documents
Lecture 28: 23 documents
Lecture 29: 20 documents
Lecture 30: 26 documents
Lecture 31: 26 documents
Lecture 32: 26 documents
Lecture 33: 23 documents
Lecture 34: 22 documents


# Step 4: Embeddings + Persistent Chroma


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import shutil

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

COLLECTION_NAME = "mit_18_06_lectures_01_34"

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)

# ------------------------------------------------------------
# Build a CLEAN 34-lecture Chroma database
# ------------------------------------------------------------

if CHROMA_DIR.exists():
    print("Removing old full-course Chroma:", CHROMA_DIR)
    shutil.rmtree(CHROMA_DIR)

CHROMA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DIR),
)

vectorstore.add_documents(documents)

print("Documents added to full-course Chroma.")
print("Collection:", COLLECTION_NAME)
print("Persistent Chroma:", CHROMA_DIR)
print("Total documents:", len(documents))

In [ ]:
# ------------------------------------------------------------
# Validate the full-course Chroma database
# ------------------------------------------------------------

stored = vectorstore.get()

print("Total vectors stored:", len(stored["ids"]))

lectures_in_chroma = sorted(
    {
        int(metadata["lecture_number"])
        for metadata in stored["metadatas"]
    }
)

print("Lectures found in Chroma:")
print(lectures_in_chroma)

assert lectures_in_chroma == list(range(1, 35)), (
    "Chroma does not contain all Lectures 1–34."
)

print("✅ All 34 lectures are present in Chroma.")

Total vectors stored: 850
Lectures found in Chroma:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
✅ All 34 lectures are present in Chroma.


# Step 5: Add CrossEncoder Reranking


In [ ]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"

reranker = CrossEncoder(
    RERANKER_MODEL
)

RETRIEVAL_K = 8
RERANK_TOP_N = 3

print("Reranker:", RERANKER_MODEL)
print("Retrieve candidates:", RETRIEVAL_K)
print("Keep after reranking:", RERANK_TOP_N)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker: cross-encoder/ms-marco-MiniLM-L6-v2
Retrieve candidates: 8
Keep after reranking: 3


In [ ]:
def retrieve_and_rerank(
    query,
    candidate_k=RETRIEVAL_K,
    top_n=RERANK_TOP_N,
    lecture_number=None,
):
    search_kwargs = {
        "k": candidate_k,
    }

    # Optional lecture-specific retrieval.
    if lecture_number is not None:
        search_kwargs["filter"] = {
            "lecture_number": int(lecture_number)
        }

    candidates = vectorstore.similarity_search(
        query,
        **search_kwargs,
    )

    if not candidates:
        return []

    pairs = [
        [query, doc.page_content]
        for doc in candidates
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidates, scores),
        key=lambda item: float(item[1]),
        reverse=True,
    )

    final_docs = []

    for rank, (doc, score) in enumerate(ranked[:top_n], start=1):
        doc.metadata["rerank_score"] = float(score)
        doc.metadata["rerank_position"] = rank
        final_docs.append(doc)

    return final_docs


In [ ]:
question = "What is the column picture?"

# Course-wide search across Lectures 1-5.
reranked_docs = retrieve_and_rerank(question)

for doc in reranked_docs:
    print(
        f"Lecture {doc.metadata['lecture_number']} | "
        f"{seconds_to_timestamp(doc.metadata['start_seconds'])} | "
        f"score={doc.metadata['rerank_score']:.3f}"
    )
    print(doc.page_content[:500])
    print()


Lecture 1 | 20:56 | score=0.549
find it. The main point is that there is, that the three planes, because they're, you know, they're not parallel, they're not special, they do meet in one point, and that's the solution. But maybe you can see that this row picture is getting a little hard to see. The row picture was a cinch when we looked at two lines meeting. When we look at three planes meeting, it's not so clear and in four dimensions, probably a little less clear. So can I quit on the row picture? I'll quit on the row pictur

Lecture 1 | 09:42 | score=0.060
this one in the right amounts to get that one. It's asking us to find the right linear combination. This is called a linear combination, and it's the most fundamental operation in the whole course. It's a linear combination of the columns. That's what we're seeing on the left side. Again, I don't want to write down a big definition. You can see what it is. There's column one, there's column two. I multiply by some numbers and I ad

# Step 6: Configure the LLM


In [ ]:
from google.colab import userdata

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("OPENAI_API_KEY loaded")
except Exception:
    print("Add OPENAI_API_KEY to Colab Secrets.")


OPENAI_API_KEY loaded


In [ ]:
from langchain_openai import ChatOpenAI

LLM_MODEL = "gpt-4o-mini"

llm = ChatOpenAI(
    model=LLM_MODEL,
    temperature=0,
)

print("LLM ready:", LLM_MODEL)


LLM ready: gpt-4o-mini


Demo LLM only, for pipeline validation — the production agent (Notebook 04) uses gpt-5.6-luna.

# Step 7: Build Multi-Lecture Reranked RAG

`ask_lecture()` searches all loaded lectures by default.

Pass `lecture_number=2` (or another lecture number) when you explicitly want lecture-filtered retrieval.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a study assistant for MIT 18.06 Linear Algebra. "
        "Answer only from the supplied reranked course evidence. "
        "If the evidence is insufficient, say you do not know. "
        "Return clean Markdown. Use short headings and bullets when helpful. "
        "Do not use decorative emphasis."
    ),
    (
        "human",
        "Question:\n{question}\n\nReranked course evidence:\n{context}"
    ),
])


def ask_lecture(question, lecture_number=None):
    docs = retrieve_and_rerank(
        question,
        lecture_number=lecture_number,
    )

    if not docs:
        return {
            "answer": "I do not know based on the indexed lecture evidence.",
            "source": None,
            "timestamp": None,
            "video_link": None,
            "docs": [],
        }

    context = "\n\n".join(
        f"SOURCE {i}\n"
        f"Lecture ID: {doc.metadata['lecture_id']}\n"
        f"Lecture {doc.metadata['lecture_number']}: "
        f"{doc.metadata['lecture_title']}\n"
        f"Timestamp: "
        f"{seconds_to_timestamp(doc.metadata['start_seconds'])}-"
        f"{seconds_to_timestamp(doc.metadata['end_seconds'])}\n"
        f"{doc.page_content}"
        for i, doc in enumerate(docs, 1)
    )

    response = llm.invoke(
        rag_prompt.format_messages(
            question=question,
            context=context,
        )
    )

    primary = docs[0]
    start = primary.metadata["start_seconds"]
    end = primary.metadata["end_seconds"]

    return {
        "answer": response.content,
        "source": (
            f"Lecture {primary.metadata['lecture_number']}: "
            f"{primary.metadata['lecture_title']}"
        ),
        "lecture_id": primary.metadata["lecture_id"],
        "lecture_number": primary.metadata["lecture_number"],
        "lecture_title": primary.metadata["lecture_title"],
        "timestamp": (
            f"{seconds_to_timestamp(start)}-"
            f"{seconds_to_timestamp(end)}"
        ),
        "video_url": primary.metadata["video_url"],
        "video_link": build_timestamp_link(
            primary.metadata["video_url"],
            start,
        ),
        "docs": docs,
    }


# Step 8: MVP Demo


In [ ]:
from IPython.display import display, Markdown

question = "What is the column picture?"

result = ask_lecture(question)

display(
    Markdown(
        result["answer"]
        + "\n\n### Source\n"
        + f"{result['source']}  \n"
        + f"Timestamp: {result['timestamp']}  \n"
        + f"[Watch this part]({result['video_link']})"
    )
)

# Example of lecture-filtered retrieval:
# result = ask_lecture("Explain elimination.", lecture_number=2)


## Column Picture

The column picture refers to a geometric interpretation of linear combinations of the columns of a matrix. Here are the key points:

- **Linear Combination**: The column picture involves taking linear combinations of the columns of a matrix to represent solutions to linear equations.
- **Visualization**: In the context of two-dimensional vectors, each column can be visualized as a vector in a plane. For example:
  - Column one might represent the vector (2, -1).
  - Column two might represent the vector (-1, 2).
- **Geometric Representation**: The goal is to find the right combination of these column vectors to achieve a specific target vector, such as (0, 3).
- **Column Space**: The set of all possible linear combinations of the columns forms what is known as the column space of the matrix.

In summary, the column picture provides a visual and conceptual framework for understanding how linear combinations of matrix columns can be used to solve linear equations.

### Source
Lecture 1: The geometry of linear equations  
Timestamp: 20:56-22:29  
[Watch this part](https://archive.org/download/MIT18.06S05_MP4/01.mp4#t=1256)

# Step 9: Save RAG Configuration for Notebook 3

Notebook 3 will later load this shared multi-lecture RAG configuration.

The configuration now describes the course-level Chroma collection rather than one lecture.


In [ ]:
rag_config = {
    "collection_name": COLLECTION_NAME,
    "persist_directory": str(CHROMA_DIR),
    "embedding_model": EMBEDDING_MODEL,
    "reranker_model": RERANKER_MODEL,
    "retrieval_k": RETRIEVAL_K,
    "rerank_top_n": RERANK_TOP_N,
    "chunk_size": CHUNK_SIZE,
    "lecture_start": LECTURE_START,
    "lecture_end": LECTURE_END,
    "lecture_numbers": LECTURE_NUMBERS,
    "lecture_ids": [
        record["lecture_id"]
        for record in lecture_records
    ],
}

with open(RAG_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(
        rag_config,
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Saved RAG config:", RAG_CONFIG_PATH)


Saved RAG config: /content/drive/MyDrive/AI_Engineering_Final_Project/rag/all_lectures_rag_config.json


## Notebook 2 Output

Notebook 2 now owns the **multi-lecture RAG architecture**:

`Lectures 1–5 STT → chunks → shared Chroma → retrieval → reranking → LLM`